# Reliable Tools and Sub-workflows

In *Tool Calling* (Ch. 7) you gave an agent a tool and let it decide when to use it. That's the fun part. This chapter is the **grown-up** part: how to make a tool you'd actually trust in production — one that talks to a real API, checks who's asking, enforces the rules in code, and asks a human before doing anything expensive.

The whole idea in one line:

> **The AI handles ambiguity. The workflow controls the deterministic rules.**

We'll build up in five moves:

1. Make a **deterministic** HTTP request (run it yourself, no AI).
2. **Validate and normalise** its inputs.
3. Wrap the deterministic logic in a **reusable sub-workflow**.
4. Expose that one **safe, high-level** sub-workflow to the agent as a tool.
5. Require **human approval** for the high-impact call.

| What you'll learn | Where it comes from |
|---|---|
| **Deterministic HTTP** — verbs, headers, body, credentials, pagination, timeouts, error codes | Deepens the HTTP tool (Ch. 7, Project 4) |
| **Safe vs. unsafe `$fromAI()`** — what the model may and may not control | Deepens Ch. 7 |
| **Sub-workflows** — explicit inputs, validation and credentials kept *inside* | New nodes |
| **Call n8n Workflow Tool** — expose one safe tool instead of many risky ones | New node |
| **Ownership checks** — verify the caller is allowed *before* acting | New |
| **Native tool approval** vs. content approval | Deepens Ch. 4d & 7 |

**Workflows in this chapter:**

| File | What it does | GitHub Link |
|------|-------------|-------------|
| `20_book_appointment_subworkflow.json` | The safe callee: validate → ownership check → write → structured result | [View](https://github.com/ezponda/ai-agents-course/blob/main/courses/n8n_no_code/book/_static/workflows/20_book_appointment_subworkflow.json) |
| `21_agent_books_via_tool.json` | The agent that calls it as one safe tool, with approval | [View](https://github.com/ezponda/ai-agents-course/blob/main/courses/n8n_no_code/book/_static/workflows/21_agent_books_via_tool.json) |

**Requirements:** an **OpenRouter API key** and **n8n 1.113+** (Data Tables). One Data Table, `appointments`. No other credentials.

---

## Move 1 — a deterministic HTTP request (before any AI)

Before you *ever* let an agent call an API, call it yourself in a plain **HTTP Request** node and get it working. An agent that wraps a broken request just fails more creatively. The HTTP Request node is the same one from Project 4 — here's the production checklist for it.

| Setting | What to know (non-technical) |
|---|---|
| **Method** | `GET` reads data; `POST` creates; `PATCH` updates; `DELETE` removes. Reads are safe to retry; writes are not (Move 5). |
| **Headers** | Extra info sent with the request (e.g. `Content-Type: application/json`). Usually set for you. |
| **Body** | The JSON you send on a `POST`/`PATCH` — the data you want to create or change. |
| **Authentication** | Pick **Predefined** (Google, Slack…) or **Generic Credential** (API key header, Bearer token). **Store the key as an n8n Credential — never paste it in the URL.** |
| **Pagination** (Options) | Big lists come in **pages**. Turn on pagination so n8n fetches page after page automatically instead of only the first 20 results. |
| **Timeout** (Options) | How long to wait before giving up. Set one (e.g. 10s) so a hung API can't freeze your workflow forever. |

**Reading the response — the status code tells you what happened:**

| Code | Meaning | What to do |
|---|---|---|
| `200` | OK | Carry on |
| `401` | Unauthenticated | Your credential is missing or wrong |
| `403` | Forbidden | Authenticated, but not *allowed* to do this (a permissions problem) |
| `404` | Not found | Wrong URL or the record doesn't exist |
| `429` | Too many requests | You hit the **rate limit** — slow down, wait, retry (see Appendix B2) |
| `500` | Server error | Their fault, not yours — retry later |

```{tip}
**Credentials live in n8n, not in the workflow.** When you create an HTTP credential, n8n stores the secret encrypted and only injects it at run time. That means you can share or export a workflow **without** leaking your keys — and the AI, which only ever sees the workflow, never sees the secret.
```

---

## Move 2 — safe vs. unsafe `$fromAI()`

`$fromAI()` (Ch. 7) lets the model fill in a value at run time — great for the *ambiguous* parts of a request (which city? which story?). But some values must **never** come from the model. The rule:

| The model MAY provide… | The model must NEVER provide… |
|---|---|
| A search term, a question | **Credentials / API keys** |
| Which record to look up (an id it was shown) | **The base URL** (it could point the tool anywhere) |
| A short, low-risk field | **User or tenant identity** (`customer_id`) — that comes from the trusted session |
| | **Permission-sensitive fields** (role, `is_admin`, price, `status`) |
| | **The whole request body** (an unrestricted body is a blank cheque) |
| | **Hard business rules** ("never refund more than paid") — those live in code |

The test is simple: *what's the worst thing that happens if the model gets this value wrong or is tricked into a bad value?* If the answer is "it charges the wrong card" or "it reads another customer's data", that value does **not** belong in `$fromAI()`. Hardcode it, take it from the trusted session, or enforce it in a node.

```{warning}
**A Code tool that runs model-written code (like Project 2's data interpreter) is the extreme case of this.** Letting the model generate and `eval` code is powerful but hands it a very sharp tool — treat it as a **security boundary**: run it with least privilege, never near real credentials or write access, and prefer a **fixed sub-workflow** (below) whenever the operation is known in advance. Most "tools" don't need arbitrary code — they need one safe, specific action.
```

---

## Moves 3 & 4 — a reusable sub-workflow, exposed as one safe tool

Here's the key architectural move. Instead of giving the agent three raw Data Table tools (find, update, delete) and *hoping* it uses them correctly, you build **one** deterministic sub-workflow — `book_appointment` — that does the whole job safely, and expose just that.

```
AI Agent
   │  calls one tool: book_appointment(appointment_id, action)
   ▼
book_appointment  (a normal n8n sub-workflow — no AI inside)
   ├─ validates the inputs
   ├─ checks the caller OWNS this appointment
   ├─ performs the write (credentials live here)
   └─ returns a small structured result { status, message }
```

Why this is better: the agent can't skip the ownership check, can't invent a `status`, and never touches the credentials — because none of that is exposed to it. The AI decides *what the customer wants*; the sub-workflow decides *what is allowed*.

**Files:** [`20_book_appointment_subworkflow.json`](https://github.com/ezponda/ai-agents-course/blob/main/courses/n8n_no_code/book/_static/workflows/20_book_appointment_subworkflow.json) (the callee) and [`21_agent_books_via_tool.json`](https://github.com/ezponda/ai-agents-course/blob/main/courses/n8n_no_code/book/_static/workflows/21_agent_books_via_tool.json) (the caller).

> **Import via URL** (both, one at a time):
> ```
> https://raw.githubusercontent.com/ezponda/ai-agents-course/main/courses/n8n_no_code/book/_static/workflows/20_book_appointment_subworkflow.json
> https://raw.githubusercontent.com/ezponda/ai-agents-course/main/courses/n8n_no_code/book/_static/workflows/21_agent_books_via_tool.json
> ```
>
> **Download:** {download}`20_book_appointment_subworkflow.json <_static/workflows/20_book_appointment_subworkflow.json>` · {download}`21_agent_books_via_tool.json <_static/workflows/21_agent_books_via_tool.json>`

```{important}
**Setup:** create a Data Table named `appointments` with **String** columns `id`, `date`, `time`, `service`, `client_name`, `customer_id`, `status`. Add a test row (e.g. `id=1`, `customer_id=alice`, `status=booked`). Import **20** first and save it, then import **21** and, in the tool node, select workflow 20.
```

::::{dropdown} 🛠️ Build the sub-workflow (20) from scratch
:color: secondary

### Step 1: New workflow + trigger
**Add Workflow** → name it `book_appointment`. Add an **Execute Sub-workflow Trigger** (the *When called* node). Set **Input data mode** to *Using fields below* and define three inputs: `customer_id`, `appointment_id`, `action` (all String). *These are the tool's contract — the only things a caller can pass.*

### Step 2: Validate the input (IF)
Add an **IF** node → `Validate input`. Two conditions (combinator **AND**): `{{ $json.customer_id }}` **is not empty** AND `{{ $json.action }}` **is not empty**. The **false** branch → an **Edit Fields** node `Return: invalid` returning `{ status: "invalid", message: "Missing required fields…" }`.

### Step 3: Look it up
On the **true** branch add a **Data Table** node → `Find appointment`, **Operation: Get Row(s)**, table `appointments`, filter `id` **equals** `{{ $('When called').item.json.appointment_id }}`.

### Step 4: The ownership check (IF)
Add an **IF** node → `Caller owns it?`: `{{ $json.customer_id }}` (from the fetched row) **equals** `{{ $('When called').item.json.customer_id }}` (from the trusted input). **False** branch → `Return: not allowed` (`status: "denied"`). *This is the whole point — a caller can only touch their own appointment.*

### Step 5: Do the write + return
On the **true** branch add a **Data Table** node → `Apply change (cancel)`, **Operation: Update Row(s)**, filter by `id`, set `status` = `cancelled`. Then an **Edit Fields** node `Return: done` → `{ status: "ok", message: "Appointment … cancelled." }`.

### Result
The sub-workflow always returns a small `{ status, message }` — success, `denied`, or `invalid` — that the agent can read and explain.
::::

### Node-by-Node — the sub-workflow (20)

| Node | Type | What it does |
|------|------|-------------|
| **When called** | Execute Sub-workflow Trigger | Receives the explicit inputs `customer_id`, `appointment_id`, `action` |
| **Validate input** | IF | Rejects the call if required fields are missing |
| **Find appointment** | Data Table (Get) | Looks up the row by `id` |
| **Caller owns it?** | IF | **Ownership check** — the row's `customer_id` must match the caller's |
| **Apply change (cancel)** | Data Table (Update) | Performs the write — only if allowed |
| **Return: done / not allowed / invalid** | Edit Fields | Returns a small structured `{ status, message }` |

### Node-by-Node — the caller (21)

| Node | Type | What it does |
|------|------|-------------|
| **When chat message received** | Chat Trigger | The customer chats |
| **Salon Assistant** | AI Agent | Collects details, confirms, calls the tool |
| **book_appointment (tool)** | Call n8n Workflow Tool | Runs sub-workflow 20 as a single tool |

```{important}
**The identity is not the model's to choose.** In the tool's input mapping, `appointment_id` and `action` come from `$fromAI(...)` (the ambiguous parts), but **`customer_id` is mapped from the trusted session** (`{{ $json.sessionId }}` here), *not* `$fromAI`. That single decision is what stops one customer cancelling another's booking — the exact gap you should always close on a write tool.
```

---

## Move 5 — human approval (and what it is *not*)

For a high-impact action you often want a person in the loop. n8n gives you **two** distinct shapes — don't confuse them:

| Shape | When | How |
|---|---|---|
| **Approve generated content** | The AI *drafts* something (an email, a reply) and a human okays it before it's sent | A **Wait** node in a normal workflow (Ch. 4d) |
| **Approve an agent's tool call** | The agent *wants to run a risky tool* (cancel, refund, delete) and a human okays that specific call | **Native tool approval** — the agent pauses and sends an Approve/Deny request (Slack/Telegram/Gmail/Chat) before the tool runs |

For our booking tool, turn on **Require Human Approval** on the `book_appointment` tool (or add a *Send and Wait* review step). Now when the agent decides to cancel, the workflow **pauses** and a person sees *exactly* which tool and which arguments — and clicks Approve or Deny.

```{warning}
**Approval confirms intent — it is not security.** A human clicking "Approve" does **not** replace authentication, authorisation, validation, or your hard business rules. Those must run in the sub-workflow **regardless** of the click. Approval is a *last* gate for high-impact actions, layered on top of the code-level checks — never instead of them. (That's why the ownership check in Move 4 runs even after approval.)
```

```{note}
**Exact UI may vary by version.** The tool-approval option and the sub-workflow input mapping have moved around across n8n releases. If a label here doesn't match your screen, check the current docs — the *shape* (one safe tool, identity from the session, approval for risky calls) is what matters, not the button's name.
```

---

## Test It / What to Observe

### 1. Cancel your own appointment
Open the chat (workflow 21). Ask to cancel appointment `1`. The assistant confirms, then (with approval on) **pauses** for your OK. Approve → the sub-workflow runs the ownership check, updates the row, and returns `status: ok`.

### 2. Try to cancel someone else's
Change the trusted `customer_id` (the tool's `customer_id` mapping) to `bob` and ask to cancel appointment `1` (owned by `alice`). **Expected:** the sub-workflow returns `status: denied` — the model never got to touch the row. *That's the ownership check doing its job.*

### 3. Watch the tool call
Open **Salon Assistant → Logs**. You'll see the single `book_appointment` call with its arguments — one clean tool, not three raw table operations.

```{important}
**This is an educational workflow.** The look-up-then-write here is **not** safe against two people cancelling at the same instant (a race condition). Making writes truly safe — idempotency, atomic operations — is the job of *Appendix B2: Production Reliability*.
```

---

## Try It Yourself

### Exercise: add a second safe action
Extend `book_appointment` to also **reschedule** (not just cancel). Add `new_date` and `new_time` inputs; when `action = 'reschedule'`, update those columns instead of `status`. Keep the **validation** and **ownership check** exactly as they are — they should protect *every* action.

**Done when:** a customer can reschedule *their own* appointment, a reschedule of someone else's still returns `denied`, and the agent still calls just **one** tool.

::::{dropdown} 🛠️ Solution sketch
:color: secondary

- Add `new_date`, `new_time` to the **Execute Sub-workflow Trigger** inputs.
- After `Caller owns it?` (true branch), add a **Switch** (or IF) on `{{ $('When called').item.json.action }}`: `cancel` → the update you already have; `reschedule` → a **Data Table Update** that sets `date` = `{{ $('When called').item.json.new_date }}` and `time` = `{{ $('When called').item.json.new_time }}`.
- Both paths end at a `Return: done`.

The ownership check sits **before** the branch, so it guards cancel *and* reschedule with no duplication — one gate, every action. That's the payoff of putting rules in the workflow instead of the prompt.
::::

---

## Summary

| Concept | What you learned |
|---------|------------------|
| **Deterministic HTTP first** | Get the request working yourself — verbs, headers, body, auth, pagination, timeouts, status codes — before an agent wraps it |
| **Credentials in n8n** | Secrets are stored encrypted and injected at run time; the model never sees them |
| **Safe `$fromAI()`** | The model fills ambiguous values only — never credentials, URLs, identity, permission fields, whole bodies, or hard rules |
| **Sub-workflow** | Explicit inputs, validation and credentials kept *inside*, a small structured result out |
| **One safe tool** | Expose one high-level `book_appointment` instead of several raw operations |
| **Ownership check** | Verify the caller owns the record *before* reading or writing it |
| **Native tool approval** | Pause for a human on high-impact calls — on top of, not instead of, code checks |

**Key ideas:**
- *The AI handles ambiguity; the workflow controls the deterministic rules.*
- Identity comes from the **trusted session**, never from `$fromAI()`.
- Approval confirms **intent** — it is not authentication, authorisation, or validation.

**Docs:**
- [HTTP Request](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.httprequest/)
- [HTTP Request credentials](https://docs.n8n.io/integrations/builtin/credentials/httprequest/)
- [Pagination](https://docs.n8n.io/code/cookbook/http-node/pagination/)
- [Sub-workflows](https://docs.n8n.io/flow-logic/subworkflows/)
- [Execute Sub-workflow](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.executeworkflow/)
- [Call n8n Workflow Tool](https://docs.n8n.io/integrations/builtin/cluster-nodes/sub-nodes/n8n-nodes-langchain.toolworkflow/)
- [Human-in-the-loop for AI tool calls](https://docs.n8n.io/advanced-ai/human-in-the-loop-tools/)